## 1. Pergunta 1 — Valor Total Recebido por Mês (Jan–Mai 2023)

**Case iFood — Data Architecture**

**Pergunta:** Qual foi o valor total recebido pelos motoristas de táxi amarelo em NYC por mês?

**Fonte de dados:** Delta Table `ifood_taxi_case.taxi_gold` — camada de consumo analítico do pipeline Lakehouse.

### a) Setup

In [0]:
%load_ext autoreload
%autoreload 2

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config.settings import Tables

print(f"Tabela: {Tables.SCHEMA}.{Tables.GOLD_TAXI}")

### b) Critério de Filtro: `total_amount > 0`

Corridas com `total_amount <= 0` representam chargebacks (`payment_type = 4 — Dispute`) e corridas sem cobrança
(`payment_type IN (3, 6) — No charge / Voided trip`). Não constituem receita efetiva para os motoristas
e são excluídas da agregação mensal.

In [0]:
%sql
SELECT
    year,
    month,
    COUNT(*)                    AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_amount_received
FROM ifood_taxi_case.taxi_gold
WHERE total_amount > 0
GROUP BY year, month
ORDER BY year, month

In [0]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df_plot = spark.sql("""
    SELECT
        month,
        ROUND(SUM(total_amount), 2) AS total_amount_received
    FROM ifood_taxi_case.taxi_gold
    WHERE total_amount > 0
    GROUP BY month
    ORDER BY month
""").toPandas()

month_labels = ["Jan", "Fev", "Mar", "Abr", "Mai"]
values = df_plot["total_amount_received"] / 1_000_000

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(month_labels, values, color="#2196F3", edgecolor="white", linewidth=0.5)

for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        f"${val:.1f}M", ha="center", va="bottom", fontsize=10, fontweight="bold"
    )

ax.set_title("Valor Total Recebido por Mês — Jan–Mai 2023", fontsize=13, pad=12)
ax.set_xlabel("Mês", fontsize=11)
ax.set_ylabel("Total Recebido (USD, milhões)", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}M"))
ax.set_ylim(0, max(values) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### c) Resultado e Interpretação

A query retorna o faturamento mensal consolidado de Jan–Mai 2023. Pontos de atenção:

- **Fevereiro** registra a menor receita do período — consistente com o menor número de dias do mês.
- **Maio** apresenta o maior faturamento acumulado — combinação de maior volume de corridas e recuperação pós-inverno.
- A tendência crescente de Jan a Mai reflete a sazonalidade típica de Nova York (inverno → primavera).

> **Visualização:** No Databricks, ao executar a célula `%sql`, selecione a aba "+" → **Bar Chart**
> com `month` no eixo X e `total_amount_received` no eixo Y.